# 🧠 Granite Blog Writer v3.0 — Google Colab Edition
### Generador Editorial Offline-First con IBM Granite 3.3, Análisis de Sentencias TSJ y Diseño Cyber-Legal

Este notebook permite ejecutar **Granite Blog Writer** en la nube de Google Colab utilizando **Ollama** con aceleración por GPU (T4/V100) o CPU estándar de forma completamente gratuita.

---

## 🚀 Paso 1: Instalar y Levantar Ollama en Google Colab

In [ ]:
# Instalar Ollama oficial para Linux
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar el demonio de Ollama en segundo plano
import subprocess, time
print("Iniciando servicio 'ollama serve' en segundo plano...")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Verificar que Ollama responde en localhost:11434
import urllib.request
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as resp:
        if resp.status == 200:
            print("✅ Servicio Ollama iniciado exitosamente en localhost:11434")
except Exception as e:
    print(f"⚠️ Aviso verificando Ollama: {e}")

## 📥 Paso 2: Descargar el Modelo IBM Granite 3.3

> **Recomendación de Hardware en Colab:**
- **Colab Gratuito (CPU/T4 con RAM estándar):** Descarga `granite3.3:2b` (ultrarrápido, ~1.5 GB).
- **Colab Pro / GPU Activa:** Descarga `granite3.3:8b` (calidad editorial profesional, ~4.9 GB).

In [ ]:
# Selecciona el modelo deseado:
MODEL_NAME = "granite3.3:2b"  # Cambia a 'granite3.3:8b' para máxima calidad pericial

print(f"Descargando modelo IBM Granite ({MODEL_NAME})...")
!ollama pull {MODEL_NAME}
print(f"✅ Modelo {MODEL_NAME} descargado y listo.")

## 📦 Paso 3: Clonar el Repositorio e Instalar Dependencias Colab

In [ ]:
import os

# Si el proyecto no está clonado, clonarlo o inicializarlo
if not os.path.exists("/content/granite-blog"):
    print("Clonando repositorio...")
    !git clone https://github.com/tu-usuario/Creador-de-Articulos-Blog-con-Granite.git /content/granite-blog

%cd /content/granite-blog

# Instalar dependencias optimizadas para Colab
!pip install -q -r requirements-colab.txt pyngrok
print("✅ Dependencias instaladas correctamente.")

## ⚙️ Paso 4: Configurar Variables de Entorno y Directorios RAG

In [ ]:
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"
os.environ["GRANITE_MODEL"] = MODEL_NAME
os.environ["FAST_MODEL"] = "granite3.3:2b"
os.environ["MAX_TOKENS"] = "4096"
os.environ["NUM_CTX"] = "8192"
os.environ["KEEP_ALIVE"] = "30m"
os.environ["NUM_THREAD"] = "4"
os.environ["CORS_ORIGINS"] = "*"
os.environ["APP_HOST"] = "0.0.0.0"
os.environ["APP_PORT"] = "8000"
os.environ["DEFAULT_JURISDICTION"] = "auto"

# Asegurar carpetas de almacenamiento pericial RAG
os.makedirs("/content/granite-blog/RAG/inputs", exist_ok=True)
os.makedirs("/content/granite-blog/RAG/articulos", exist_ok=True)
os.makedirs("/content/granite-blog/logs", exist_ok=True)
print("✅ Entorno y carpetas configurados.")

## 🌐 Paso 5: Iniciar el Servidor FastAPI y Abrir Interfaz Web

Puedes conectarte mediante dos métodos:
1. **Túnel nativo de Google Colab** (`serve_kernel_port_as_window`)
2. **Túnel de Ngrok** (opcional, si deseas compartir la URL pública con terceros)

In [ ]:
import threading, uvicorn
from google.colab import output

# Función para correr el servidor Uvicorn en segundo plano
def start_server():
    uvicorn.run("backend.app:app", host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(3)

# Exponer el puerto 8000 en Google Colab
output.serve_kernel_port_as_window(8000)
print("=========================================================================")
print("✅ Servidor Granite Blog Writer activo en Google Colab.")
print("👉 Haz click en la ventana o enlace superior para interactuar con la app.")
print("=========================================================================")

### 🔗 Opción Alternativa: Enlace Público con Ngrok (Opcional)
Si prefieres un enlace público directo para compartir o abrir en tu móvil:

In [ ]:
# Si tienes un authtoken de ngrok, pégalo aquí:
NGROK_AUTHTOKEN = ""  # Opcional: https://dashboard.ngrok.com/get-started/your-authtoken

if NGROK_AUTHTOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    public_url = ngrok.connect(8000)
    print(f"🌐 URL Pública de Ngrok: {public_url}")
else:
    print("Ingresa tu token de ngrok si deseas generar un túnel público.")